# 05 — Data products

A **data product** in Digicities is anything shippable that's been packaged alongside its RDF description: a weather file, a timeseries dump, a building stock CSV. The platform consumes them by reading their TTL "manifest" and pointing code at the referenced data files.

This notebook covers:

1. Two patterns for parsing TTL back into Python: **rdflib/SPARQL directly** and the **`TTLParser`** convenience class
2. When to use which
3. Round-tripping sample data end-to-end (file → dict → further processing)

In [ ]:
import os, sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve().parent))
os.environ.setdefault("TRIPLESTORE_BACKEND", "fuseki")
os.environ.setdefault("GRAPHDB_URL", "http://localhost:3030")

import rdflib
from rdflib.namespace import RDF, RDFS

## 5.1 Direct parsing with rdflib + SPARQL

This pattern works on **any** TTL. Load the file into an in-memory `rdflib.Graph`, then query it with SPARQL — same dialect you use against GraphDB. For small files (sample data, fixtures, tests) this is usually enough.

In [ ]:
g = rdflib.Graph()
g.parse("sample_data/alpine_village.ttl", format="turtle")
print(f"Loaded {len(g)} triples into in-memory graph")

In [ ]:
q = """
    PREFIX dici_onto: <https://digicities.info/ontology#>
    PREFIX qudt:      <http://qudt.org/schema/qudt/>
    PREFIX rdfs:      <http://www.w3.org/2000/01/rdf-schema#>

    SELECT ?comp_label ?attr_label ?value ?unit WHERE {
      ?comp rdfs:label ?comp_label ;
            dici_onto:hasAttribute ?attr .
      ?attr rdfs:label ?attr_label ;
            qudt:value ?value .
      OPTIONAL { ?attr dici_onto:hasUnitLabel ?unit }
      FILTER(STRSTARTS(STR(?comp), 'https://digicities.info/tutorial/alpine_village/'))
    } ORDER BY ?comp_label ?attr_label
"""

# rdflib returns tuples directly; turn them into a DataFrame
import pandas as pd
rows = [(str(c), str(a), str(v), str(u) if u else "") for c, a, v, u in g.query(q)]
pd.DataFrame(rows, columns=["component", "attribute", "value", "unit"]).head(10)

## 5.2 The `TTLParser` convenience class

`backend.data_products.TTLParser` targets the **specific TTL format the Streamlit Replica Builder's Excel importer produces**. It encodes a bunch of heuristics about:

- Which URIs are components vs. attributes
- How to unpack Physical / Cost / Curve / Dynamic attribute shapes
- How to turn QUDT unit IRIs back into strings

If you're reading TTL that came out of the Excel importer or the UI, use it — you get structured component dicts for free. If you're reading arbitrary TTL, prefer rdflib directly.

A minimal round-trip:

In [ ]:
from backend.data_products import TTLParser

parser = TTLParser()
with open("sample_data/alpine_village.ttl", "r", encoding="utf-8") as f:
    graph = parser.parse_ttl_content(f.read())

print(f"TTLParser loaded: {len(graph)} triples")
print("Known attribute types it recognises:")
for t in sorted(parser.KNOWN_ATTRIBUTE_TYPES):
    print(f"  {t}")

`parser.extract_components_from_graph(graph)` groups by component type. It assumes the TTL was generated by the Excel importer — specifically, that attribute URIs are sub-URIs of the component (e.g. `BuildingA/ElectricityDemand`) and linked via named predicates like `dici_onto:hasBuildingElectricityDemandAttribute`.

Our Alpine Village TTL uses the simpler generic `dici_onto:hasAttribute` predicate, which the parser doesn't unpack, so you'll get a different view. That's a signal to use rdflib directly — which is exactly what notebook 03 did to build the scenario dict.

In [ ]:
extracted = parser.extract_components_from_graph(graph)
# What types did it successfully identify?
{ctype: len(comps) for ctype, comps in extracted.items()}

## 5.3 Resource references — pointing at data files

Data products often carry pointers into file storage: `dici_onto:storedAt` (a path in NextCloud or the local filesystem) on a `TimeSeries` or `Resource` attribute. The TTL parser exposes these as `component['resources'][attr_name]` so you can resolve them later.

Since the Alpine Village sample has no timeseries references, here's a tiny inline TTL snippet to demonstrate the shape:

In [ ]:
snippet = '''
@prefix dici_onto: <https://digicities.info/ontology#> .
@prefix qudt:      <http://qudt.org/schema/qudt/> .
@prefix unit:      <http://qudt.org/vocab/unit/> .
@prefix rdfs:      <http://www.w3.org/2000/01/rdf-schema#> .
@prefix xsd:       <http://www.w3.org/2001/XMLSchema#> .
@prefix ex:        <https://digicities.info/tutorial/data_product/> .

ex:WeatherFile a dici_onto:TimeSeries ;
    rdfs:label "Hourly irradiance — Alpine Valley 2024"^^xsd:string ;
    dici_onto:storedAt "workspace_demo/data_products/weather/alpine_valley_2024.csv"^^xsd:string ;
    qudt:unit unit:W-PER-M2 ;
    dici_onto:hasUnitLabel "W/m²"^^xsd:string .
'''

mini = rdflib.Graph()
mini.parse(data=snippet, format="turtle")

DICI = rdflib.Namespace("https://digicities.info/ontology#")
for ts, path in mini.subject_objects(DICI.storedAt):
    print(f"{ts}\n  → {path}")

## Summary

- **rdflib + SPARQL** is the always-works answer. Use it for parsing any TTL where you know the shape you want.
- **`TTLParser`** is a convenience layer over rdflib tuned for UI-generated data-product TTL. Worth reaching for when you're consuming output from the Streamlit replica/data-product modules.
- Data products conventionally carry a `dici_onto:storedAt` pointer into file storage. Keep your data and its RDF manifest in the same workspace directory; the UI expects that layout.

Next: [`06_api_submission.ipynb`](06_api_submission.ipynb) - handing a scenario to the bundled demo energy simulator and reading the results back.